In [2]:
import json
import pandas as pd

INPUT_FILE = "02_guided_assigned.json"
OUTPUT_FILE = "02_guided_sample_600.json"

N_SAMPLES = 600
RANDOM_STATE = 42

# ============================================================
# 1. LOAD DATA
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Tổng số mẫu:", len(df))

print("\nDomain:")
print(df["domain"].value_counts())

print("\nQuestion type:")
print(df["question_type"].value_counts())

print("\nRequested label:")
print(df["requested_label"].value_counts())


# ============================================================
# 2. TẠO STRATUM
# domain × question_type × requested_label
# ============================================================

stratify_cols = [
    "domain",
    "question_type",
    "requested_label"
]

# Số lượng từng nhóm
group_sizes = (
    df.groupby(stratify_cols, dropna=False)
      .size()
      .reset_index(name="n")
)

print("\nSố strata:", len(group_sizes))
print(group_sizes)


# ============================================================
# 3. PHÂN BỔ 600 SAMPLE
# Tỷ lệ theo kích thước group nhưng mỗi group có ít nhất 1 sample
# ============================================================

group_sizes["raw"] = (
    group_sizes["n"] / group_sizes["n"].sum() * N_SAMPLES
)

group_sizes["sample_n"] = (
    group_sizes["raw"]
    .round()
    .astype(int)
    .clip(lower=1)
)

# Không lấy nhiều hơn số sample thực tế của group
group_sizes["sample_n"] = group_sizes[
    ["sample_n", "n"]
].min(axis=1)


# ============================================================
# 4. ĐIỀU CHỈNH CHÍNH XÁC VỀ 600
# ============================================================

while group_sizes["sample_n"].sum() > N_SAMPLES:

    candidates = group_sizes[
        group_sizes["sample_n"] > 1
    ].sort_values("sample_n", ascending=False)

    if len(candidates) == 0:
        break

    idx = candidates.index[0]
    group_sizes.loc[idx, "sample_n"] -= 1


while group_sizes["sample_n"].sum() < N_SAMPLES:

    candidates = group_sizes[
        group_sizes["sample_n"] < group_sizes["n"]
    ].copy()

    if len(candidates) == 0:
        break

    # ưu tiên group đang thiếu nhiều nhất
    candidates["remaining"] = (
        candidates["n"] - candidates["sample_n"]
    )

    idx = candidates.sort_values(
        "remaining",
        ascending=False
    ).index[0]

    group_sizes.loc[idx, "sample_n"] += 1


print(
    "\nTổng sample sẽ lấy:",
    group_sizes["sample_n"].sum()
)


# ============================================================
# 5. SAMPLE TỪNG STRATUM
# ============================================================

sampled_parts = []

for _, row in group_sizes.iterrows():

    mask = pd.Series(True, index=df.index)

    for col in stratify_cols:
        value = row[col]

        if pd.isna(value):
            mask &= df[col].isna()
        else:
            mask &= df[col].eq(value)

    group = df[mask]

    part = group.sample(
        n=int(row["sample_n"]),
        random_state=RANDOM_STATE
    )

    sampled_parts.append(part)


sampled = pd.concat(
    sampled_parts,
    ignore_index=True
)

# Shuffle lần cuối
sampled = sampled.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


# ============================================================
# 6. KIỂM TRA PHÂN BỐ
# ============================================================

print("\n===== FINAL SAMPLE =====")
print("Total:", len(sampled))

print("\nDomain:")
print(sampled["domain"].value_counts())

print("\nQuestion type:")
print(sampled["question_type"].value_counts())

print("\nRequested label:")
print(sampled["requested_label"].value_counts())

print("\nDomain × Question type:")
print(
    pd.crosstab(
        sampled["domain"],
        sampled["question_type"]
    )
)


# ============================================================
# 7. CHỈ GIỮ FIELD CẦN THIẾT
# ============================================================

keep_fields = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question_type",
    "question",
    "answer",
    "requested_label"
]

sampled_short = sampled[keep_fields]


# ============================================================
# 8. SAVE
# ============================================================

sampled_short.to_json(
    OUTPUT_FILE,
    orient="records",
    force_ascii=False,
    indent=2
)

print("\nSaved:", OUTPUT_FILE)

Tổng số mẫu: 14883

Domain:
domain
news         2853
science      2744
finance      2567
medical      2339
education    2244
legal        2136
Name: count, dtype: int64

Question type:
question_type
vanilla          7006
multi_hop        2304
false_premise    1589
unanswerable     1455
set_operation    1365
comparison       1164
Name: count, dtype: int64

Requested label:
requested_label
intrinsic_hallucination        4961
mixed_partial_hallucination    4961
extrinsic_hallucination        4961
Name: count, dtype: int64

Số strata: 96
       domain  question_type              requested_label    n
0   education     comparison      extrinsic_hallucination   56
1   education     comparison      intrinsic_hallucination   57
2   education     comparison  mixed_partial_hallucination   80
3   education  false_premise      extrinsic_hallucination   71
4   education  false_premise      intrinsic_hallucination   87
..        ...            ...                          ...  ...
91    science  set_

In [3]:
import json
import re
import random
from collections import Counter


INPUT_FILE = "02_guided_sample_600.json"
OUTPUT_FILE = "02_guided_sample_600_target.json"

random.seed(42)


# ============================================================
# 1. ƯỚC LƯỢNG SỐ FACTUAL SPAN / CLAIM TRONG GOLD ANSWER
# ============================================================

def split_claims(answer):
    """
    Tách Gold Answer thành các factual span/claim tương đối độc lập.

    Ví dụ:
    "Doanh thu là 5.000 tỷ và lợi nhuận là 2.000 tỷ."
        -> 2 spans

    "10 phút."
        -> 1 span
    """

    if not answer:
        return []

    text = str(answer).strip()

    # Tách theo dấu câu mạnh
    parts = re.split(
        r"[.;!?]+",
        text
    )

    claims = []

    for part in parts:

        part = part.strip(" ,:-")

        if not part:
            continue

        # Tách thêm các cấu trúc liệt kê / nhiều ý
        subparts = re.split(
            r"\s*;\s*|"
            r",\s*(?=(?:đồng thời|ngoài ra|trong khi|còn|và hiện|và được|và có|"
            r"đạt|giảm|tăng|phục hồi|loại bỏ|điều trị|phát triển))",
            part,
            flags=re.IGNORECASE
        )

        for sub in subparts:
            sub = sub.strip(" ,:-")

            # bỏ fragment quá ngắn vô nghĩa
            if len(sub.split()) >= 2:
                claims.append(sub)

    return claims


# ============================================================
# 2. KIỂM TRA CÓ ĐỦ ĐIỀU KIỆN MIXED KHÔNG
# ============================================================

def can_be_mixed(answer):
    """
    Mixed cần ít nhất 2 factual spans:
    - ít nhất 1 span giữ đúng
    - ít nhất 1 span để hallucinate
    """

    claims = split_claims(answer)

    return len(claims) >= 2


# ============================================================
# 3. GÁN TARGET LABEL
# ============================================================

def assign_target_label(item):

    requested = item["requested_label"]
    answer = item.get("answer", "")

    # intrinsic / extrinsic giữ nguyên
    if requested in {
        "intrinsic_hallucination",
        "extrinsic_hallucination"
    }:
        return requested

    # mixed chỉ khi có >= 2 factual spans
    if requested in {
        "mixed_partial_hallucination",
        "mixed_hallucination"
    }:

        if can_be_mixed(answer):
            return "mixed_partial_hallucination"

        # chỉ có 1 factual span -> không đủ cấu trúc mixed
        # chuyển ngẫu nhiên sang intrinsic/extrinsic
        return random.choice([
            "intrinsic_hallucination",
            "extrinsic_hallucination"
        ])

    raise ValueError(
        f"Unknown requested_label: {requested}"
    )


# ============================================================
# 4. LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)


# ============================================================
# 5. PROCESS
# ============================================================

output = []

for item in data:

    claims = split_claims(
        item.get("answer", "")
    )

    target = assign_target_label(item)

    row = {
        **item,

        # số factual span ước lượng
        "gold_span_count": len(claims),

        # để inspect nếu cần
        "gold_spans": claims,

        # label cuối dùng để generate
        "target_hallucination_label": target
    }

    output.append(row)


# ============================================================
# 6. SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 7. SUMMARY
# ============================================================

print("Total:", len(output))

print("\nRequested label:")
print(
    Counter(
        x["requested_label"]
        for x in output
    )
)

print("\nFinal target label:")
print(
    Counter(
        x["target_hallucination_label"]
        for x in output
    )
)


changed = [
    x for x in output
    if x["requested_label"]
    != x["target_hallucination_label"]
]

print("\nMixed bị đổi vì không đủ 2 span:", len(changed))


for x in changed[:10]:

    print("\n", x["chunk_id"])

    print("Answer:")
    print(x["answer"])

    print("Spans:")
    print(x["gold_spans"])

    print(
        x["requested_label"],
        "->",
        x["target_hallucination_label"]
    )

Total: 600

Requested label:
Counter({'mixed_partial_hallucination': 200, 'intrinsic_hallucination': 200, 'extrinsic_hallucination': 200})

Final target label:
Counter({'intrinsic_hallucination': 292, 'extrinsic_hallucination': 269, 'mixed_partial_hallucination': 39})

Mixed bị đổi vì không đủ 2 span: 161

 finance_sbv_87678_chunk13
Answer:
Đảng ủy cơ quan NHTW chỉ đạo thực hiện nghiêm túc việc đẩy mạnh học tập và làm theo tư tưởng, đạo đức, phong cách Hồ Chí Minh trong toàn Đảng bộ.
Spans:
['Đảng ủy cơ quan NHTW chỉ đạo thực hiện nghiêm túc việc đẩy mạnh học tập và làm theo tư tưởng, đạo đức, phong cách Hồ Chí Minh trong toàn Đảng bộ']
mixed_partial_hallucination -> intrinsic_hallucination

 news_dantri_74598_chunk3
Answer:
2,5 km.
Spans:
['2,5 km']
mixed_partial_hallucination -> intrinsic_hallucination

 education_apd_56424_chunk9
Answer:
10 phút.
Spans:
['10 phút']
mixed_partial_hallucination -> extrinsic_hallucination

 science_vjst_44224_chunk16
Answer:
Họ được phỏng vấn về tình h

In [5]:
import json


INPUT_FILE = "02_guided_sample_600_target.json"
OUTPUT_FILE = "02_guided_sample_600_clean.json"

KEEP_KEYS = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question",
    "answer",
    "target_hallucination_label",
]


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


clean_data = []

for item in data:
    clean_item = {
        key: item.get(key)
        for key in KEEP_KEYS
    }
    clean_data.append(clean_item)


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        clean_data,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Total:", len(clean_data))
print("Saved:", OUTPUT_FILE)

Total: 600
Saved: 02_guided_sample_600_clean.json


In [7]:
import json

INPUT_FILE = "data/02_guided_sample_600_clean.json"
OUTPUT_FILE = "data/02_guided_sample_600_final.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    if item["target_hallucination_label"] == "mixed_partial_hallucination":
        item["target_hallucination_label"] = "mixed_hallucination"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", OUTPUT_FILE)

Saved: data/02_guided_sample_600_final.json


In [8]:
import pandas as pd

df = pd.read_json("data/output_qwen/03_qwen_3_8_flash_600.json")
len(df)

600

In [3]:
import json
from pathlib import Path

INPUT_FILE = Path("data/output_gpt/03_gpt5_mini_600.json")
OUTPUT_FILE = Path("data/output_gpt/03_gpt5_mini_retry_input.json")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

retry_data = []

for sample in data:
    generation = sample.get("gpt_generation")

    failed = False

    # Không có block generation
    if not isinstance(generation, dict):
        failed = True
    else:
        result = generation.get("result")
        meta = generation.get("meta") or {}

        # Có lỗi trong meta
        if meta.get("error"):
            failed = True

        # result = null
        elif result is None:
            failed = True

        # result không đúng dạng
        elif not isinstance(result, dict):
            failed = True

        # status khác ok
        elif result.get("status") != "ok":
            failed = True

        # thiếu hallucinated_response
        elif not isinstance(result.get("hallucinated_response"), str):
            failed = True

        elif not result.get("hallucinated_response").strip():
            failed = True

    if failed:
        retry_data.append({
            "chunk_id": sample.get("chunk_id"),
            "doc_id": sample.get("doc_id"),
            "domain": sample.get("domain"),
            "content": sample.get("content"),
            "question": sample.get("question"),
            "answer": sample.get("answer"),
            "target_hallucination_label": sample.get(
                "target_hallucination_label"
            ),
        })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        retry_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Total input: {len(data)}")
print(f"Retry samples: {len(retry_data)}")
print(f"Saved to: {OUTPUT_FILE}")

Total input: 600
Retry samples: 87
Saved to: data/output_gpt/03_gpt5_mini_retry_input.json


In [5]:
from pathlib import Path

BASE_DIR = Path.cwd()

print(BASE_DIR)

/home/trang/study/khoaluan/KLTN/src/create_hallu/qa


In [7]:
import json
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

ORIGINAL_FILE = Path(
    "data/output_gpt/03_gpt5_mini_600.json"
)

FIX_FILE = Path(
    "data/output_gpt/output_gpt_fix/03_gpt5_mini_600.json"
)

OUTPUT_FILE = Path(
    "data/judge_input/gpt5_mini_judge_input.json"
)


# ============================================================
# LOAD
# ============================================================

with open(ORIGINAL_FILE, "r", encoding="utf-8") as f:
    original_data = json.load(f)

with open(FIX_FILE, "r", encoding="utf-8") as f:
    fix_data = json.load(f)

print("Original:", len(original_data))
print("Fix     :", len(fix_data))


# ============================================================
# KEY
# chunk_id có thể trùng -> phải dùng question
# ============================================================

def make_key(item):
    return (
        item.get("chunk_id"),
        item.get("question"),
    )


# ============================================================
# VALID GENERATION
# ============================================================

def is_valid(item):
    generation = item.get("gpt_generation")

    if not isinstance(generation, dict):
        return False

    result = generation.get("result")

    if not isinstance(result, dict):
        return False

    # Không đưa not_applicable / invalid_gold vào Judge
    if result.get("status") != "ok":
        return False

    response = result.get("hallucinated_response")

    if not isinstance(response, str):
        return False

    if not response.strip():
        return False

    return True


# ============================================================
# FIX MAP
# ============================================================

fix_map = {
    make_key(item): item
    for item in fix_data
}


# ============================================================
# MERGE + CREATE JUDGE INPUT
# ============================================================

judge_input = []

replaced = 0
original_used = 0
excluded = 0

for original in original_data:

    key = make_key(original)

    # Mặc định dùng generation cũ
    selected = original

    # Nếu có retry và retry thành công -> dùng bản retry
    if key in fix_map and is_valid(fix_map[key]):
        selected = fix_map[key]
        replaced += 1

    # Sau merge vẫn không có response hợp lệ -> bỏ
    if not is_valid(selected):
        excluded += 1
        continue

    result = selected["gpt_generation"]["result"]

    # ========================================================
    # CHỈ ĐƯA THÔNG TIN JUDGE CẦN
    # KHÔNG đưa target_label / perturbation_plan / spans
    # ========================================================

    judge_input.append({
        "chunk_id": selected.get("chunk_id"),
        "doc_id": selected.get("doc_id"),
        "domain": selected.get("domain"),

        "context": selected.get("content"),
        "question": selected.get("question"),
        "gold_answer": selected.get("answer"),

        "hallucinated_answer": result.get(
            "hallucinated_response"
        ),

        "generator": "gpt5_mini",
    })

    if selected is original:
        original_used += 1


# ============================================================
# DUPLICATE CHECK
# ============================================================

keys = [
    (x["chunk_id"], x["question"])
    for x in judge_input
]

duplicate_count = len(keys) - len(set(keys))


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        judge_input,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("JUDGE INPUT SUMMARY")
print("=" * 60)

print(f"Original samples      : {len(original_data)}")
print(f"Retry samples         : {len(fix_data)}")
print(f"Replaced by retry     : {replaced}")
print(f"Used original         : {original_used}")
print(f"Excluded              : {excluded}")
print(f"Judge input           : {len(judge_input)}")
print(f"Duplicate QA          : {duplicate_count}")

print("\nSaved to:")
print(OUTPUT_FILE)

Original: 600
Fix     : 87

JUDGE INPUT SUMMARY
Original samples      : 600
Retry samples         : 87
Replaced by retry     : 67
Used original         : 513
Excluded              : 20
Judge input           : 580
Duplicate QA          : 0

Saved to:
data/judge_input/gpt5_mini_judge_input.json


In [8]:
import json
from pathlib import Path

files = [
    "gpt5_mini_judge_input.json",
    "deepseek_v4_flash_judge_input.json",
    "qwen_3_8_flash_judge_input.json",
]

for filename in files:
    path = Path("data/judge_input") / filename

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"{filename}: {len(data)} samples")

gpt5_mini_judge_input.json: 580 samples
deepseek_v4_flash_judge_input.json: 576 samples
qwen_3_8_flash_judge_input.json: 590 samples


# is true no hallu

In [1]:
import json
from pathlib import Path

RESULT_DIR = Path("generated_hallu_detector_results")

FILES = [
    "gpt5_mini_no_hallu.json",
    "deepseek_v4_flash_no_hallu.json",
    "qwen_3_8_flash_no_hallu.json",
]

for filename in FILES:
    path = RESULT_DIR / filename

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        item["is_true_no_hallu"] = None

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Updated {filename}: {len(data)} samples")

Updated gpt5_mini_no_hallu.json: 75 samples
Updated deepseek_v4_flash_no_hallu.json: 117 samples
Updated qwen_3_8_flash_no_hallu.json: 138 samples


FileNotFoundError: File data/gpt5.6_luna_ground_truth_final.json does not exist